<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet%2014%20PBL/Praktikum%201%20(JS14)%20PBL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langkah 1 - Import Library

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

#Langkah 2 - Pra Pengolahan Data (Data Preprocessing)

In [5]:
# Memuat dataset cafes.csv dan reviews.csv
df_cafes = pd.read_csv('cafes.csv')
df_reviews = pd.read_csv('reviews.csv')

# Menggabungkan data berdasarkan cafe_id
df_merged = pd.merge(df_reviews, df_cafes, on='cafe_id', how='inner', suffixes=('_review', '_cafe'))

# Tentukan target regresi (misal: rating kafe) dan bersihkan data
target_column = 'rating_cafe' if 'rating_cafe' in df_merged.columns else 'rating'
df_merged = df_merged.dropna(subset=[target_column])

# Pilih beberapa kolom fitur numerik untuk direpresentasikan ke citra
feature_cols = ['review_count', 'latitude', 'longitude']
df_features = df_merged[feature_cols].fillna(0)

# Normalisasi nilai fitur ke rentang [0, 255] agar dapat direpresentasikan sebagai piksel gambar
scaler = MinMaxScaler(feature_range=(0, 255))
scaled_features = scaler.fit_transform(df_features)

# Mengubah setiap baris data tabular menjadi citra dummy berukuran (64, 64, 3)
def tabular_to_image(row_data):
    # Buat kanvas kosong 64x64x3
    img = np.zeros((64, 64, 3), dtype=np.uint8)
    # Petakan nilai fitur ke channel warna RGB gambar
    val1 = int(np.clip(row_data[0], 0, 255))
    val2 = int(np.clip(row_data[1] * 10 + 128, 0, 255)) # penyesuaian skala latitude
    val3 = int(np.clip(row_data[2] * 10 + 128, 0, 255)) # penyesuaian skala longitude

    img[:, :, 0] = val1
    img[:, :, 1] = val2
    img[:, :, 2] = val3
    return img

images = np.array([tabular_to_image(row) for row in scaled_features])
labels = df_merged[target_column].values

# Membagi data menjadi training dan testing (analog train_set / test_set)
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)

# Normalisasi nilai piksel citra menjadi antara 0 dan 1 (setara rescale = 1./255)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"Bentuk X_train (Data Training Citra): {X_train.shape}")
print(f"Bentuk X_test  (Data Testing Citra) : {X_test.shape}")

Bentuk X_train (Data Training Citra): (1784, 64, 64, 3)
Bentuk X_test  (Data Testing Citra) : (446, 64, 64, 3)


#Langkah 2 - Pra Pengolahan Data & Konversi ke Citra (Dataset Cafes & Reviews)

In [4]:
# Memuat dataset cafes.csv dan reviews.csv
df_cafes = pd.read_csv('cafes.csv')
df_reviews = pd.read_csv('reviews.csv')

# Menggabungkan data berdasarkan cafe_id
df_merged = pd.merge(df_reviews, df_cafes, on='cafe_id', how='inner', suffixes=('_review', '_cafe'))

# Tentukan target regresi (misal: rating kafe) dan bersihkan data
target_column = 'rating_cafe' if 'rating_cafe' in df_merged.columns else 'rating'
df_merged = df_merged.dropna(subset=[target_column])

# Pilih beberapa kolom fitur numerik untuk direpresentasikan ke citra
feature_cols = ['review_count', 'latitude', 'longitude']
df_features = df_merged[feature_cols].fillna(0)

# Normalisasi nilai fitur ke rentang [0, 255] agar dapat direpresentasikan sebagai piksel gambar
scaler = MinMaxScaler(feature_range=(0, 255))
scaled_features = scaler.fit_transform(df_features)

# Mengubah setiap baris data tabular menjadi citra dummy berukuran (64, 64, 3)
def tabular_to_image(row_data):
    # Buat kanvas kosong 64x64x3
    img = np.zeros((64, 64, 3), dtype=np.uint8)
    # Petakan nilai fitur ke channel warna RGB gambar
    val1 = int(np.clip(row_data[0], 0, 255))
    val2 = int(np.clip(row_data[1] * 10 + 128, 0, 255)) # penyesuaian skala latitude
    val3 = int(np.clip(row_data[2] * 10 + 128, 0, 255)) # penyesuaian skala longitude

    img[:, :, 0] = val1
    img[:, :, 1] = val2
    img[:, :, 2] = val3
    return img

images = np.array([tabular_to_image(row) for row in scaled_features])
labels = df_merged[target_column].values

# Membagi data menjadi training dan testing (analog train_set / test_set)
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)

# Normalisasi nilai piksel citra menjadi antara 0 dan 1 (setara rescale = 1./255)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"Bentuk X_train (Data Training Citra): {X_train.shape}")
print(f"Bentuk X_test  (Data Testing Citra) : {X_test.shape}")

Bentuk X_train (Data Training Citra): (1784, 64, 64, 3)
Bentuk X_test  (Data Testing Citra) : (446, 64, 64, 3)


#Langkah 3 - Pembuatan Model CNN

#Langkah 3.1 - Inisiasi Model CNN

In [6]:
cnn = tf.keras.models.Sequential()

#Langkah 3.2 - Pembuatan Layer Konvolusi 1

In [7]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


#Langkah 3.3 - Pembuatan Layer Pooling 1

In [8]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

#Langkah 3.4 - Pembuatan Layer Konvolusi 2 dan Pooling 2

In [9]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

#Langkah 3.5 - Flattening

In [10]:
cnn.add(tf.keras.layers.Flatten())

#Langkah 3.6 - Fully Connected Layer 1 (Input)

In [11]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

#Langkah 3.7 - Fully Connected Layer 2 (Output untuk Regresi)

In [12]:
# Menggunakan 1 unit tanpa aktivasi (atau linear) karena ini kasus regresi nilai kontinu (misal: prediksi rating)
cnn.add(tf.keras.layers.Dense(units=1))

#Langkah 3.8 - Compile Model CNN

In [13]:
# Menggunakan loss function mean_squared_error (MSE) karena ini adalah kasus regresi
cnn.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       802,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 813,217 (3.10 MB)

 Trainable params: 813,217 (3.10 MB)

 Non-trainable params: 0 (0.00 B)

#Langkah 4 - Fit CNN

In [14]:
history = cnn.fit(
    x=X_train,
    y=y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 8s 118ms/step - loss: 0.4713 - mae: 0.3409 - val_loss: 0.0401 - val_mae: 0.1544
Epoch 2/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 5s 88ms/step - loss: 0.0335 - mae: 0.1436 - val_loss: 0.0538 - val_mae: 0.1846
Epoch 3/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 116ms/step - loss: 0.0304 - mae: 0.1370 - val_loss: 0.0266 - val_mae: 0.1261
Epoch 4/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 5s 98ms/step - loss: 0.0354 - mae: 0.1471 - val_loss: 0.0313 - val_mae: 0.1469
Epoch 5/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 9s 85ms/step - loss: 0.0292 - mae: 0.1326 - val_loss: 0.0297 - val_mae: 0.1380
Epoch 6/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 116ms/step - loss: 0.0488 - mae: 0.1703 - val_loss: 0.0459 - val_mae: 0.1682
Epoch 7/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 5s 97ms/step - loss: 0.0385 - mae: 0.1589 - val_loss: 0.0408 - val_mae: 0.1565
Epoch 8/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 99ms/step - loss: 0.0275 - mae: 0.1281 - val_loss: 0.0273 - val_mae: 0.1271
Epoch 9/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 5s 84ms/step - loss: 0.

#Langkah 5 - Prediksi dengan 1 Citra Data Kafe

In [15]:
# Mengambil 1 sampel citra dari data testing
test_image = X_test[0:1] # mempertahankan dimensi batch (1, 64, 64, 3)
actual_value = y_test[0]

# Melakukan prediksi dengan model CNN
result = cnn.predict(test_image)
predicted_rating = result[0][0]

print("Hasil Prediksi Model CNN untuk 1 Data Kafe:")
print(f"- Nilai Aktual   : {actual_value}")
print(f"- Hasil Prediksi : {predicted_rating:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Hasil Prediksi Model CNN untuk 1 Data Kafe:
- Nilai Aktual   : 4.7
- Hasil Prediksi : 4.7539
